# LeetCode 1033 — Moving Stones Until Consecutive bằng Python

Link đề: https://leetcode.com/problems/moving-stones-until-consecutive/ · Độ khó: **Medium** · Chủ đề: Math (toán), Brainteaser (đố mẹo)

Skeleton LeetCode:

```python
class Solution:
    def numMovesStones(self, a: int, b: int, c: int) -> List[int]:
        ...
```

Ghi chú: có ba viên đá ở ba vị trí **khác nhau** trên trục số. Mỗi nước đi: nhấc viên đá ở **đầu mút** (thấp nhất hoặc cao nhất) và đặt vào một **vị trí nguyên còn trống nằm giữa hai đầu mút**. Trò chơi kết thúc khi ba viên đá nằm ở ba vị trí **liên tiếp**. Trả về `[số nước đi ít nhất, số nước đi nhiều nhất]`. Đáp án là cặp **duy nhất** nên so sánh đúng cặp. Chỉ có công thức `O(1)` sau khi sắp xếp: nước đi nhiều nhất bằng `z - x - 2` (mỗi nước đi rút ngắn khoảng cách hai đầu mút đúng `1`), còn nước đi ít nhất chỉ có thể là `0`, `1` hoặc `2`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trạng thái là bộ ba vị trí, nước đi hợp lệ, điều kiện dừng, và hai số cần tìm (ít nhất và nhiều nhất).
2. Nắm các công cụ Python cần thiết: `sorted`, giải nén `x, y, z = sorted((a, b, c))`, đệ quy có nhớ với `lru_cache`, và mô phỏng nước đi bằng hàm kiểm tra hợp lệ.
3. Hiểu nguyên lý: sau khi sắp xếp `x < y < z`, mỗi nước đi làm khoảng cách hai đầu mút `z - x` giảm **ít nhất 1**, từ đó cận `max = z - x - 2`; và vì chỉ cần một hoặc hai nước đi để "gom" ba viên, `min` chỉ nhận các giá trị `0`, `1`, `2`.
4. Đi qua 4 cách: tìm kiếm toàn bộ cây trò chơi bằng đệ quy có nhớ (brute force, oracle), mô phỏng từng nước đi chậm nhất, công thức `O(1)` (cách nộp chính) và dựng **dãy nước đi cụ thể** rồi kiểm tra bằng bộ mô phỏng.
5. Tránh các lỗi kinh điển: quên **sắp xếp** `a, b, c`, lệch một đơn vị trong `z - x - 2`, chỉ xét khoảng trống bên trái (quên bên phải), dùng `<` thay `<=` cho khoảng trống bằng `2`, quên trường hợp đã liên tiếp (`0`), và trả `[max, min]` sai thứ tự.

## 2. Đề bài & ví dụ

Có ba viên đá ở các vị trí khác nhau trên trục `X`. Cho ba số nguyên `a`, `b`, `c` là vị trí các viên đá.

Trong **một nước đi**, bạn nhấc một viên đá ở **đầu mút** (vị trí thấp nhất hoặc cao nhất) và đặt nó vào một vị trí nguyên **còn trống** nằm **giữa** hai đầu mút. Chính xác hơn, nếu các viên đang ở `x < y < z`, bạn nhấc viên ở `x` hoặc `z` và đặt nó tại vị trí nguyên `k` với `x < k < z` và `k != y`.

Trò chơi kết thúc khi không thể đi tiếp (ba viên ở ba vị trí liên tiếp).

Trả về mảng `answer` độ dài `2`: `answer[0]` là **số nước đi ít nhất** có thể, `answer[1]` là **số nước đi nhiều nhất** có thể.

**Ví dụ 1:** `a = 1, b = 2, c = 5` -> `[1, 2]`. Ít nhất: chuyển `5` sang `3` (một nước). Nhiều nhất: `5 -> 4`, rồi `4 -> 3` (hai nước).

**Ví dụ 2:** `a = 4, b = 3, c = 2` -> `[0, 0]`. Không đi được nước nào.

**Ví dụ 3:** `a = 3, b = 5, c = 1` -> `[1, 2]`. Ít nhất: chuyển `1` sang `4`; nhiều nhất: `1 -> 2` rồi `2 -> 4`.

**Ràng buộc:**

- `1 <= a, b, c <= 100`
- `a`, `b`, `c` khác nhau đôi một.

In [1]:
from typing import List, Tuple, Dict
from functools import lru_cache
from itertools import combinations, permutations
import random, time

examples = [
    ((1, 2, 5), [1, 2]),
    ((4, 3, 2), [0, 0]),
    ((3, 5, 1), [1, 2]),
]
for (a, b, c), expected in examples:
    print(f"a={a} b={b} c={c} -> kỳ vọng {expected}")

a=1 b=2 c=5 -> kỳ vọng [1, 2]
a=4 b=3 c=2 -> kỳ vọng [0, 0]
a=3 b=5 c=1 -> kỳ vọng [1, 2]


## 3. Kiến thức nền

### 3.1. Sắp xếp rồi giải nén

Vì nước đi chỉ phụ thuộc vào **thứ tự trên trục**, ta đổi `(a, b, c)` thành `x < y < z` bằng `x, y, z = sorted((a, b, c))`. Đề cho `a, b, c` theo thứ tự bất kỳ (ví dụ 2 và 3), nên bước này là **bắt buộc**.

In [2]:
for t in ((4, 3, 2), (3, 5, 1), (1, 2, 5)):
    x, y, z = sorted(t)
    print(t, "->", (x, y, z), "| khoảng cách các đầu mút z - x =", z - x, "| hai khoảng trống: y - x - 1 =", y - x - 1, ", z - y - 1 =", z - y - 1)

(4, 3, 2) -> (2, 3, 4) | khoảng cách các đầu mút z - x = 2 | hai khoảng trống: y - x - 1 = 0 , z - y - 1 = 0
(3, 5, 1) -> (1, 3, 5) | khoảng cách các đầu mút z - x = 4 | hai khoảng trống: y - x - 1 = 1 , z - y - 1 = 1
(1, 2, 5) -> (1, 2, 5) | khoảng cách các đầu mút z - x = 4 | hai khoảng trống: y - x - 1 = 0 , z - y - 1 = 2


### 3.2. Nước đi hợp lệ và điều kiện dừng

Từ `x < y < z`: chọn đầu mút `x` hoặc `z`, đặt vào `k` với `x < k < z` và `k != y`. Có nước đi hợp lệ ⇔ tồn tại `k` như vậy ⇔ `z - x > 2` (khoảng giữa có ít nhất `2` vị trí nguyên: `y` và một vị trí trống). Khi `z - x == 2` thì `y` là vị trí duy nhất ở giữa: ba viên liên tiếp, **hết nước đi**.

In [3]:
def legal_moves(state: Tuple[int, int, int]):
    x, y, z = sorted(state)
    moves = []
    for src in (x, z):
        for k in range(x + 1, z):
            if k != y:
                moves.append((src, k))
    return moves

for s in ((1, 2, 3), (1, 2, 5), (1, 4, 6)):
    print(s, "->", len(legal_moves(s)), "nước đi hợp lệ:", legal_moves(s)[:6])

(1, 2, 3) -> 0 nước đi hợp lệ: []
(1, 2, 5) -> 4 nước đi hợp lệ: [(1, 3), (1, 4), (5, 3), (5, 4)]
(1, 4, 6) -> 6 nước đi hợp lệ: [(1, 2), (1, 3), (1, 5), (6, 2), (6, 3), (6, 5)]


### 3.3. Mỗi nước đi rút ngắn khoảng cách đầu mút

Nếu nhấc `z` đặt vào `k` (`x < k < z`, `k != y`) thì đầu mút mới là `max(y, k)` nhỏ hơn `z` ít nhất `1`; tương tự khi nhấc `x`. Vậy `z - x` **giảm ít nhất `1`** sau mỗi nước đi. Trò chơi kết thúc khi `z - x = 2`: nên số nước đi **tối đa** là `(z - x) - 2`, đạt được khi mỗi nước đi chỉ làm `z - x` giảm đúng `1`.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `a = 3, b = 5, c = 1` -> sắp xếp `(x, y, z) = (1, 3, 5)`. Khoảng cách `z - x = 4`.

- **Nhiều nhất** = `z - x - 2 = 2`: `1 -> 2` (được `2, 3, 5`), rồi `2 -> 4` (được `3, 4, 5`): hai nước.
- **Ít nhất**: chỉ cần `1` nước vì khoảng trống bên trái hay bên phải bằng `1`: `y - x = 2`: ta đặt viên `z` vào vị trí trống giữa `x` và `y`; chính xác `1 -> 4` đặt `x` vào khoảng giữa `y` và `z` được `3, 4, 5`: một nước.

Ví dụ 1: `(1, 2, 5)`: `y - x = 1` (kề nhau) ⇒ đặt `5` vào `3` được `1, 2, 3`: `1` nước. Nhiều nhất `5 - 1 - 2 = 2`.

Ví dụ 2: `(2, 3, 4)` liên tiếp: `[0, 0]`.

Trường hợp **hai khoảng trống đều lớn**, ví dụ `(1, 4, 9)`: không thể xong trong một nước (đặt một viên đầu mút vào cạnh viên kia thì vẫn còn viên thứ ba cách xa); cần `2` nước (đưa `1` vào sát `4`: `3, 4, 9`; rồi `9` vào sát `4`: `3, 4, 5`).

## 5. Cách 1 — Brute Force: tìm kiếm toàn bộ cây trò chơi

**Ý tưởng.** Trạng thái là bộ ba `(x, y, z)` đã sắp xếp. Từ mỗi trạng thái có tối đa `2 * (z - x - 2)` nước đi; không còn nước đi thì cả `min` lẫn `max` bằng `0`; ngược lại `min = 1 + min(mọi trạng thái kế tiếp)` và `max = 1 + max(...)`. Do `z - x` giảm ngặt nên đệ quy dừng. Dùng `lru_cache` để không tính lại. Không dùng công thức nào nên đây là **oracle độc lập** (xét **mọi** nước đi hợp lệ).

In [4]:
@lru_cache(None)
def solve_state(x: int, y: int, z: int) -> Tuple[int, int]:
    """(ít nhất, nhiều nhất) số nước đi từ trạng thái x < y < z, theo đúng luật của đề."""
    best_min, best_max = None, None
    for src in (x, z):
        for k in range(x + 1, z):
            if k == y:
                continue
            rest = tuple(sorted(({x, y, z} - {src}) | {k}))
            lo, hi = solve_state(*rest)
            best_min = lo + 1 if best_min is None else min(best_min, lo + 1)
            best_max = hi + 1 if best_max is None else max(best_max, hi + 1)
    return (0, 0) if best_min is None else (best_min, best_max)

def moves_brute(a: int, b: int, c: int) -> List[int]:
    return list(solve_state(*sorted((a, b, c))))

for (a, b, c), expected in examples:
    got = moves_brute(a, b, c)
    print(f"({a},{b},{c}): brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

(1,2,5): brute = [1, 2] OK
(4,3,2): brute = [0, 0] OK
(3,5,1): brute = [1, 2] OK


**Độ phức tạp và nút thắt.** Số trạng thái `C(N, 3)` với `N` là độ rộng vị trí (`~ 161700` khi `N = 100`), mỗi trạng thái `O(N)` nước đi: `~ 10^7` đến `10^8` thao tác, và đệ quy sâu tới `N`. Đo thử:

In [5]:
for N in (10, 20, 30, 40):
    solve_state.cache_clear()
    t0 = time.perf_counter()
    moves_brute(1, 1 + N // 2 + 1, N)
    states = solve_state.cache_info().currsize
    print(f"N={N:3}: {states:6} trạng thái, {(time.perf_counter() - t0) * 1000:9.1f} ms")

N= 10:     57 trạng thái,       0.4 ms
N= 20:    392 trạng thái,       4.2 ms
N= 30:   1252 trạng thái,      22.3 ms
N= 40:   2887 trạng thái,      48.2 ms


Nút thắt: ta duyệt hàng nghìn trạng thái mà đáp án chỉ phụ thuộc vào **hai khoảng cách** `y - x` và `z - y`. Ta tìm công thức.

## 6. Ý tưởng thuật toán / bước đột phá

Đặt `g1 = y - x - 1` và `g2 = z - y - 1` là số vị trí trống ở hai khoảng. Khi đó `z - x = g1 + g2 + 2`.

- **Nhiều nhất:** mỗi nước đi giảm `z - x` đúng `1` (ví dụ nhấc đầu mút và đặt nó **sát** đầu mút mới kề viên giữa phía bên trong, hoặc sát chính nó); cận trên đạt được, nên `max = z - x - 2 = g1 + g2`.
- **Ít nhất:**
  - `0` nếu đã liên tiếp (`g1 = g2 = 0`);
  - `1` nếu **một** trong hai khoảng trống nhỏ (`g1 <= 1` hoặc `g2 <= 1`, và chưa liên tiếp): đặt viên đầu mút **xa** vào khoảng trống nhỏ hoặc sát viên giữa (nếu `g = 0`: đặt nó vào cạnh nhóm hai viên kề nhau, cần `k` nằm trong khoảng bên kia; nếu `g = 1`: đặt nó đúng vào ô trống duy nhất);
  - `2` còn lại (cả hai khoảng trống `>= 2`): cần đưa từng viên đầu mút sát viên giữa.

| Cách | Ý tưởng |
|---|---|
| 2 | Mô phỏng nước đi chậm nhất (mỗi nước rút ngắn đúng `1`) để đếm `max` |
| 3 | Công thức `O(1)` theo `x, y, z` (nộp) |
| 4 | Dựng **dãy nước đi cụ thể** cho `min` và `max`, kiểm tra bằng bộ mô phỏng |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mô phỏng nước đi chậm nhất để đếm `max`

**Ý tưởng khác.** Để biết `max` ta có thể **chơi** chậm nhất: mỗi nước đi chỉ rút ngắn khoảng cách đúng `1`. Cụ thể từ `x < y < z` (chưa liên tiếp): nếu `z - 1 != y` thì chuyển `z` sang `z - 1`; ngược lại chuyển `x` sang `x + 1`. Đếm số nước tới khi liên tiếp.

**Giải thích từng đoạn.**

- `z - 1 != y`: vị trí `z - 1` còn trống nên đặt `z` vào đó (đầu mút mới là `max(y, z - 1) = z - 1`, giảm đúng `1`).
- Ngược lại `z - 1 == y` (viên giữa sát `z`): chuyển `x` sang `x + 1`; khi đó `x + 1 != y` vì nếu `x + 1 == y` thì đã liên tiếp.
- Vòng lặp dừng khi `z - x == 2`.

In [6]:
def simulate_max(a: int, b: int, c: int) -> int:
    x, y, z = sorted((a, b, c))
    count = 0
    while z - x > 2:
        if z - 1 != y:
            z -= 1                    # đặt viên cao nhất sát xuống một ô
        else:
            x += 1                    # viên giữa sát z: đẩy viên thấp nhất lên một ô
        count += 1
    return count

for (a, b, c), expected in examples:
    got = simulate_max(a, b, c)
    print(f"({a},{b},{c}): max mô phỏng = {got}", "OK" if got == expected[1] else "SAI")
    assert got == expected[1]

(1,2,5): max mô phỏng = 2 OK
(4,3,2): max mô phỏng = 0 OK
(3,5,1): max mô phỏng = 2 OK


**Độ phức tạp (Cách 2):** `O(z - x)` thời gian (tối đa `98` bước), `O(1)` bộ nhớ. Chỉ cho `max`; phần `min` vẫn cần phân tích trường hợp.

**Code nộp LeetCode (Cách 2, ghép với công thức cho `min`):**

In [7]:
class Solution:
    def numMovesStones(self, a: int, b: int, c: int) -> List[int]:
        x, y, z = sorted((a, b, c))
        # nhiều nhất: mô phỏng nước đi chậm nhất
        count = 0
        lo, hi = x, z
        while hi - lo > 2:
            if hi - 1 != y:
                hi -= 1
            else:
                lo += 1
            count += 1
        # ít nhất
        if z - x == 2:
            least = 0
        elif y - x <= 2 or z - y <= 2:
            least = 1
        else:
            least = 2
        return [least, count]

sol_simulate = Solution().numMovesStones
for (a, b, c), expected in examples:
    assert sol_simulate(a, b, c) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


---

### 7.2. Cách 3 — Công thức `O(1)` (cách nộp chính)

**Giải thích từng đoạn.**

- `x, y, z = sorted((a, b, c))`.
- `max = z - x - 2`.
- `min`: nếu `z - x == 2` thì `0` (đã liên tiếp); ngược lại nếu `y - x <= 2` hoặc `z - y <= 2` thì `1`; ngược lại `2`.
- Chú ý thứ tự kiểm tra: phải xét `z - x == 2` **trước** (khi đã liên tiếp thì `y - x = 1 <= 2` cũng thỏa nhánh `1`).

In [8]:
def moves_formula(a: int, b: int, c: int) -> List[int]:
    x, y, z = sorted((a, b, c))
    if z - x == 2:
        least = 0
    elif y - x <= 2 or z - y <= 2:
        least = 1
    else:
        least = 2
    return [least, z - x - 2]

for (a, b, c), expected in examples:
    got = moves_formula(a, b, c)
    print(f"({a},{b},{c}): công thức = {got}", "OK" if got == expected else "SAI")
    assert got == expected

(1,2,5): công thức = [1, 2] OK
(4,3,2): công thức = [0, 0] OK
(3,5,1): công thức = [1, 2] OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def numMovesStones(self, a: int, b: int, c: int) -> List[int]:
        x, y, z = sorted((a, b, c))                 # đưa về x < y < z
        if z - x == 2:                              # đã liên tiếp
            least = 0
        elif y - x <= 2 or z - y <= 2:              # một khoảng trống bằng 0 hoặc 1
            least = 1
        else:
            least = 2
        return [least, z - x - 2]

sol_formula = Solution().numMovesStones
for (a, b, c), expected in examples:
    assert sol_formula(a, b, c) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(1)` thời gian và bộ nhớ.

---

### 7.3. Cách 4 — Dựng dãy nước đi cụ thể và kiểm tra

**Ý tưởng khác.** Thay vì chỉ tính số, ta **dựng** một dãy nước đi cụ thể đạt `min` và một dãy đạt `max`, rồi dùng **bộ mô phỏng** kiểm tra từng nước hợp lệ và trạng thái cuối liên tiếp. Dãy `min` có độ dài đúng `min` (chứng minh `min` đạt được), cùng với phần chứng minh cận dưới ở mục 10. Dãy nước đi không duy nhất nên ta dùng **hàm kiểm tra** (mô phỏng) thay vì so sánh với một dãy cố định.

**Giải thích từng đoạn.**

- `apply_move(state, src, k)`: kiểm tra `src` là đầu mút, `k` nằm giữa hai đầu mút và chưa bị chiếm; trả về trạng thái mới đã sắp xếp.
- `min_moves_plan`: theo các trường hợp ở mục 6.
- `max_moves_plan`: mỗi nước rút ngắn đúng `1` như Cách 2.

In [10]:
State = Tuple[int, int, int]

def apply_move(state: State, src: int, k: int) -> State:
    x, y, z = state
    assert src in (x, z), "chỉ được nhấc viên ở đầu mút"
    assert x < k < z and k != y, "chỉ được đặt vào vị trí trống nằm giữa hai đầu mút"
    return tuple(sorted(({x, y, z} - {src}) | {k}))

def is_consecutive(state: State) -> bool:
    return state[2] - state[0] == 2

def run_plan(a: int, b: int, c: int, plan: List[Tuple[int, int]]) -> int:
    """Mô phỏng dãy nước đi (src, k): kiểm tra hợp lệ, trạng thái cuối phải liên tiếp; trả về số nước."""
    state = tuple(sorted((a, b, c)))
    for src, k in plan:
        state = apply_move(state, src, k)
    assert is_consecutive(state), ("chưa liên tiếp ở cuối", state)
    return len(plan)

def min_moves_plan(a: int, b: int, c: int) -> List[Tuple[int, int]]:
    x, y, z = sorted((a, b, c))
    if z - x == 2:
        return []
    if y - x == 2:
        return [(z, x + 1)]                 # đặt z vào ô trống duy nhất giữa x và y
    if y - x == 1:
        return [(z, y + 1)]                 # x, y kề nhau: đặt z sát y
    if z - y == 2:
        return [(x, y + 1)]                 # đặt x vào ô trống giữa y và z
    if z - y == 1:
        return [(x, y - 1)]                 # y, z kề nhau: đặt x sát y
    return [(x, y - 1), (z, y + 1)]         # cả hai khoảng trống >= 2: đưa từng đầu mút sát viên giữa

def max_moves_plan(a: int, b: int, c: int) -> List[Tuple[int, int]]:
    x, y, z = sorted((a, b, c))
    plan = []
    while z - x > 2:
        if z - 1 != y:
            plan.append((z, z - 1)); z -= 1
        else:
            plan.append((x, x + 1)); x += 1
    return plan

def moves_by_plans(a: int, b: int, c: int) -> List[int]:
    return [run_plan(a, b, c, min_moves_plan(a, b, c)), run_plan(a, b, c, max_moves_plan(a, b, c))]

for (a, b, c), expected in examples:
    got = moves_by_plans(a, b, c)
    print(f"({a},{b},{c}): dựng dãy = {got}", "OK" if got == expected else "SAI")
    assert got == expected

(1,2,5): dựng dãy = [1, 2] OK
(4,3,2): dựng dãy = [0, 0] OK
(3,5,1): dựng dãy = [1, 2] OK


**Code nộp LeetCode (Cách 4):** (cùng kết quả; không cần bộ mô phỏng kiểm tra khi nộp, ở đây chỉ trả về độ dài các dãy dựng được)

In [11]:
class Solution:
    def numMovesStones(self, a: int, b: int, c: int) -> List[int]:
        x, y, z = sorted((a, b, c))
        if z - x == 2:
            least = 0
        elif y - x <= 2 or z - y <= 2:
            least = 1                     # đặt viên đầu mút xa vào ô trống / sát cặp kề nhau
        else:
            least = 2                     # đưa từng đầu mút sát viên giữa
        most = 0
        while z - x > 2:                  # mỗi nước chỉ rút ngắn khoảng cách đúng 1
            if z - 1 != y:
                z -= 1
            else:
                x += 1
            most += 1
        return [least, most]

sol_plan = Solution().numMovesStones
for (a, b, c), expected in examples:
    assert sol_plan(a, b, c) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(z - x)` thời gian (dựng dãy `max`), `O(z - x)` bộ nhớ cho dãy; công thức `O(1)` vẫn đủ khi chỉ cần số lượng.

## 8. TRACE từng bước

### 8.1. Cách 2/4 cho `(1, 4, 9)`

In từng nước đi của dãy `max` và dãy `min`.

In [12]:
def trace_plan(a, b, c, plan, label):
    state = tuple(sorted((a, b, c)))
    print(f"{label}: bắt đầu {state}")
    for src, k in plan:
        state = apply_move(state, src, k)
        print(f"   nhấc {src} đặt vào {k} -> {state}")
    print(f"   kết thúc {state}, liên tiếp: {is_consecutive(state)}, số nước = {len(plan)}")
    return len(plan)

lo = trace_plan(1, 4, 9, min_moves_plan(1, 4, 9), "dãy ÍT NHẤT")
hi = trace_plan(1, 4, 9, max_moves_plan(1, 4, 9), "dãy NHIỀU NHẤT")
assert (lo, hi) == (2, 6) == tuple(moves_formula(1, 4, 9))

dãy ÍT NHẤT: bắt đầu (1, 4, 9)
   nhấc 1 đặt vào 3 -> (3, 4, 9)
   nhấc 9 đặt vào 5 -> (3, 4, 5)
   kết thúc (3, 4, 5), liên tiếp: True, số nước = 2
dãy NHIỀU NHẤT: bắt đầu (1, 4, 9)
   nhấc 9 đặt vào 8 -> (1, 4, 8)
   nhấc 8 đặt vào 7 -> (1, 4, 7)
   nhấc 7 đặt vào 6 -> (1, 4, 6)
   nhấc 6 đặt vào 5 -> (1, 4, 5)
   nhấc 1 đặt vào 2 -> (2, 4, 5)
   nhấc 2 đặt vào 3 -> (3, 4, 5)
   kết thúc (3, 4, 5), liên tiếp: True, số nước = 6


### 8.2. Bảng giá trị: ít nhất theo hai khoảng trống

In `min` theo `(g1, g2)` với `0 <= g1, g2 <= 4` (cột `g1`, hàng `g2`); `0` chỉ ở `(0, 0)`.

In [13]:
print("g2\\g1 " + " ".join(f"{g1:2}" for g1 in range(5)))
for g2 in range(5):
    row = []
    for g1 in range(5):
        x, y, z = 1, 1 + g1 + 1, 1 + g1 + 1 + g2 + 1
        row.append(f"{moves_formula(x, y, z)[0]:2}")
    print(f"  {g2}    " + " ".join(row))
assert moves_formula(1, 2, 3)[0] == 0 and moves_formula(1, 3, 4)[0] == 1 and moves_formula(1, 5, 9)[0] == 2

g2\g1  0  1  2  3  4
  0     0  1  1  1  1
  1     1  1  1  1  1
  2     1  1  2  2  2
  3     1  1  2  2  2
  4     1  1  2  2  2


### 8.3. Vì sao phải sắp xếp trước

Với `(3, 5, 1)` nếu coi `a = 3` là vị trí thấp nhất ta tính sai khoảng cách. Sắp xếp cho `(1, 3, 5)`.

In [14]:
a, b, c = 3, 5, 1
print("không sắp xếp: c - a - 2 =", c - a - 2, "| sau khi sắp xếp: z - x - 2 =", max(a, b, c) - min(a, b, c) - 2)
assert Solution().numMovesStones(a, b, c) == [1, 2]

không sắp xếp: c - a - 2 = -4 | sau khi sắp xếp: z - x - 2 = 2


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (công thức `O(1)`): ngắn gọn và dễ chứng minh.

```python
class Solution:
    def numMovesStones(self, a: int, b: int, c: int) -> List[int]:
        x, y, z = sorted((a, b, c))                 # đưa về x < y < z
        if z - x == 2:                              # đã liên tiếp
            least = 0
        elif y - x <= 2 or z - y <= 2:              # một khoảng trống bằng 0 hoặc 1
            least = 1
        else:
            least = 2
        return [least, z - x - 2]
```

In [15]:
class Solution:
    def numMovesStones(self, a: int, b: int, c: int) -> List[int]:
        x, y, z = sorted((a, b, c))                 # đưa về x < y < z
        if z - x == 2:                              # đã liên tiếp
            least = 0
        elif y - x <= 2 or z - y <= 2:              # một khoảng trống bằng 0 hoặc 1
            least = 1
        else:
            least = 2
        return [least, z - x - 2]

sol = Solution()
for (a, b, c), expected in examples:
    got = sol.numMovesStones(a, b, c)
    print(f"({a},{b},{c}): {got}", "OK" if got == expected else "SAI")
    assert got == expected

(1,2,5): [1, 2] OK
(4,3,2): [0, 0] OK
(3,5,1): [1, 2] OK


## 10. Vì sao đúng

Sắp xếp `x < y < z`, `g1 = y - x - 1`, `g2 = z - y - 1`.

**Nhiều nhất.** Mỗi nước đi làm đầu mút mới nằm trong `[x + 1, z - 1]`: nếu nhấc `z` đặt vào `k`, các đầu mút mới là `x` và `max(y, k) <= z - 1`, nên `z - x` giảm `>= 1`; nhấc `x` tương tự. Trò chơi dừng khi `z - x = 2`. Vậy tối đa `z - x - 2` nước. Cận này đạt được bằng chiến lược ở Cách 2: mỗi nước chỉ làm `z - x` giảm đúng `1` (nếu `z - 1 != y` chuyển `z` sang `z - 1`; ngược lại `z - 1 = y` và `x + 1 != y` nên chuyển `x` sang `x + 1`; trường hợp `x + 1 = y` và `z - 1 = y` nghĩa là đã liên tiếp).

**Ít nhất.**

- `0` ⇔ đã liên tiếp (không có nước đi hợp lệ).
- `1`: **đạt được** nếu `y - x <= 2` hoặc `z - y <= 2` (và chưa liên tiếp), vì: (i) `y - x == 2`: đặt `z` vào ô trống duy nhất `x + 1` (hợp lệ vì `x < x + 1 < z`, `x + 1 != y`) được `x, x + 1, y` liên tiếp; (ii) `y - x == 1`: đặt `z` vào `y + 1` (hợp lệ vì `y + 1 < z` do chưa liên tiếp, `z - y >= 2`) được `x, y, y + 1`; (iii) `z - y <= 2` đối xứng.
- `1` **là cận dưới** khi chưa liên tiếp (cần ít nhất một nước đi).
- **Một nước không đủ** khi `y - x >= 3` và `z - y >= 3`: nếu nhấc `z` đặt vào `k` thì ba viên là `x, y, k`, trải dài ít nhất từ `x` đến `y` tức `>= 3 > 2` nên chưa liên tiếp; nếu nhấc `x` đặt vào `k` thì ba viên là `k, y, z` trải dài `>= z - y >= 3`: cũng chưa liên tiếp. Vậy `min >= 2` trong trường hợp này.
- `2` **đạt được** khi cả hai khoảng trống `>= 2`: nhấc `x` đặt vào `y - 1` (hợp lệ), được `y - 1, y, z` với `z - y >= 3 > 2`, rồi nhấc `z` đặt vào `y + 1` (hợp lệ vì `y + 1 < z`), được `y - 1, y, y + 1` liên tiếp.

**Kiểm chứng bằng máy.** Oracle (Cách 1) xét **mọi** nước đi hợp lệ theo luật của đề (đệ quy có nhớ) và cho `(min, max)` thật sự. Ta so với công thức trên **mọi** bộ ba khác nhau trong `[1, 24]` (theo mọi thứ tự), cho bốn cách; rồi trên toàn miền `[1, 100]` so công thức với mô phỏng, và kiểm tra các dãy nước đi dựng ở Cách 4 đều hợp lệ.

In [16]:
solve_state.cache_clear()
count = 0
for combo in combinations(range(1, 25), 3):
    for perm in permutations(combo):
        ref = moves_brute(*perm)
        assert moves_formula(*perm) == sol.numMovesStones(*perm) == sol_simulate(*perm) == sol_plan(*perm) == moves_by_plans(*perm) == ref, perm
        count += 1
print("đã đối chiếu", count, "bộ (a, b, c) khác nhau trong [1, 24] (mọi thứ tự): bốn cách khớp oracle")

đã đối chiếu 12144 bộ (a, b, c) khác nhau trong [1, 24] (mọi thứ tự): bốn cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Cây trò chơi có nhớ | `O(N^3 * N)` với `N` là độ rộng vị trí | `O(N^3)` |
| 2. Mô phỏng nước chậm nhất + phân tích `min` | `O(z - x)` | `O(1)` |
| 3. Công thức | `O(1)` | `O(1)` |
| 4. Dựng dãy nước đi | `O(z - x)` | `O(z - x)` |

Đo ở cỡ tối đa (`1, 50, 100` và các cặp cực biên), lặp nhiều lần:

In [17]:
cases = [(1, 50, 100), (1, 2, 100), (1, 99, 100), (1, 3, 100), (98, 99, 100)]
for name, fn, reps in (("Cách 2", sol_simulate, 20000), ("Cách 3", sol_formula, 200000), ("Cách 4", sol_plan, 20000), ("nộp", sol.numMovesStones, 200000)):
    t0 = time.perf_counter()
    for _ in range(reps):
        for case in cases:
            fn(*case)
    dt = (time.perf_counter() - t0) / reps / len(cases) * 1e6
    print(f"{name:7}: {dt:8.3f} µs/lần")
print([sol.numMovesStones(*c) for c in cases])

Cách 2 :    5.103 µs/lần


Cách 3 :    0.398 µs/lần


Cách 4 :    4.554 µs/lần


nộp    :    0.287 µs/lần
[[2, 97], [1, 97], [1, 97], [1, 97], [0, 0]]


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, toàn bộ miền ràng buộc `[1, 100]`, và các tính chất. Đáp án là **cặp duy nhất** nên so sánh **chính xác**.

In [18]:
def all_solutions():
    return {"formula": moves_formula, "simulate": sol_simulate, "plan": sol_plan, "final": sol.numMovesStones}

def check_all(a, b, c, expected=None, brute=True):
    results = {name: tuple(fn(a, b, c)) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, ((a, b, c), results)
    ref = list(results["formula"])
    if brute and max(a, b, c) <= 40:
        assert moves_brute(a, b, c) == ref, (a, b, c, moves_brute(a, b, c), ref)
    if expected is not None:
        assert ref == expected, ((a, b, c), ref, expected)
    return ref

# 12.1 ví dụ đề
for (a, b, c), expected in examples:
    check_all(a, b, c, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ((1, 2, 3), [0, 0]),          # đã liên tiếp
    ((3, 1, 2), [0, 0]),          # liên tiếp nhưng không theo thứ tự
    ((98, 99, 100), [0, 0]),      # liên tiếp ở biên trên
    ((1, 2, 4), [1, 1]),          # z - x = 3: chỉ có thể đặt 4 -> 3 (một nước); max = 3 - 2 = 1
    ((1, 3, 4), [1, 1]),          # đối xứng: đặt 1 vào 2
    ((1, 3, 5), [1, 2]),          # hai khoảng trống bằng 1: min 1 (đặt 1 vào 4 hoặc 5 vào 2), max = 5 - 1 - 2 = 2
    ((1, 2, 5), [1, 2]),          # ví dụ 1
    ((1, 4, 5), [1, 2]),          # đối xứng ví dụ 1
    ((1, 4, 7), [2, 4]),          # hai khoảng trống bằng 2: min 2, max = 7 - 1 - 2 = 4
    ((1, 4, 9), [2, 6]),          # tính tay ở bài tập 2
    ((1, 50, 100), [2, 97]),      # rộng: min 2, max = 99 - 2 = 97
    ((1, 2, 100), [1, 97]),       # hai viên kề nhau, viên thứ ba xa: min 1, max = 97
    ((1, 99, 100), [1, 97]),      # đối xứng
    ((1, 3, 100), [1, 97]),       # y - x = 2: min 1
    ((1, 4, 100), [2, 97]),       # y - x = 3, z - y lớn: min 2
    ((1, 98, 100), [1, 97]),      # z - y = 2: min 1
    ((1, 97, 100), [2, 97]),      # z - y = 3, y - x lớn: min 2
    ((7, 3, 11), [2, 6]),         # thứ tự bất kỳ: sắp xếp (3, 7, 11): hai khoảng trống 3
]
for (a, b, c), expected in edge:
    check_all(a, b, c, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 18 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle trên [1, 12])
def wrong_no_sort(a, b, c):                       # quên sắp xếp: coi a < b < c
    x, y, z = a, b, c
    if z - x == 2:
        least = 0
    elif y - x <= 2 or z - y <= 2:
        least = 1
    else:
        least = 2
    return [least, z - x - 2]

def wrong_max_off_by_one(a, b, c):                # max = z - x - 1
    x, y, z = sorted((a, b, c))
    r = moves_formula(a, b, c)
    return [r[0], z - x - 1]

def wrong_left_gap_only(a, b, c):                 # chỉ xét khoảng trống bên trái
    x, y, z = sorted((a, b, c))
    if z - x == 2:
        least = 0
    elif y - x <= 2:
        least = 1
    else:
        least = 2
    return [least, z - x - 2]

def wrong_strict_less(a, b, c):                   # dùng < 2 thay vì <= 2 cho khoảng trống
    x, y, z = sorted((a, b, c))
    if z - x == 2:
        least = 0
    elif y - x < 2 or z - y < 2:
        least = 1
    else:
        least = 2
    return [least, z - x - 2]

def wrong_consecutive_last(a, b, c):              # xét nhánh "1" trước khi xét đã liên tiếp
    x, y, z = sorted((a, b, c))
    if y - x <= 2 or z - y <= 2:
        least = 1
    elif z - x == 2:
        least = 0
    else:
        least = 2
    return [least, z - x - 2]

def wrong_swapped(a, b, c):                       # trả về [max, min]
    r = moves_formula(a, b, c)
    return [r[1], r[0]]

def find_counter(wrong, limit=12):
    for combo in combinations(range(1, limit + 1), 3):
        for perm in permutations(combo):
            if wrong(*perm) != moves_brute(*perm):
                return perm
    return None

counters = {}
for name, fn in (("quên sắp xếp", wrong_no_sort), ("max lệch một đơn vị", wrong_max_off_by_one),
                 ("chỉ xét khoảng trống bên trái", wrong_left_gap_only), ("dùng < 2 thay vì <= 2", wrong_strict_less),
                 ("xét nhánh 1 trước nhánh đã liên tiếp", wrong_consecutive_last), ("trả về [max, min]", wrong_swapped)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    print(f"{name:38} (a, b, c)={case}: sai -> {fn(*case)}, đúng -> {moves_brute(*case)}")
    check_all(*case)
print("12.3 phản ví dụ: OK")

quên sắp xếp                           (a, b, c)=(1, 3, 2): sai -> [1, -1], đúng -> [0, 0]
max lệch một đơn vị                    (a, b, c)=(1, 2, 3): sai -> [0, 1], đúng -> [0, 0]
chỉ xét khoảng trống bên trái          (a, b, c)=(1, 4, 5): sai -> [2, 2], đúng -> [1, 2]
dùng < 2 thay vì <= 2                  (a, b, c)=(1, 3, 5): sai -> [2, 2], đúng -> [1, 2]
xét nhánh 1 trước nhánh đã liên tiếp   (a, b, c)=(1, 2, 3): sai -> [1, 0], đúng -> [0, 0]
trả về [max, min]                      (a, b, c)=(1, 2, 5): sai -> [2, 1], đúng -> [1, 2]
12.3 phản ví dụ: OK


In [20]:
# 12.4 toàn miền ràng buộc [1, 100]: các cách nhanh với nhau (công thức, mô phỏng, dãy nước đi hợp lệ) trên mọi tổ hợp, đủ thứ tự cho mẫu
cnt = 0
for x, y, z in combinations(range(1, 101), 3):
    ref = moves_formula(x, y, z)
    assert sol_simulate(x, y, z) == sol_plan(x, y, z) == ref
    cnt += 1
print("12.4 toàn miền [1, 100]:", cnt, "bộ tổ hợp: công thức = mô phỏng = dãy nước đi")

rng = random.Random(4033)
for _ in range(20000):
    a, b, c = rng.sample(range(1, 101), 3)
    assert moves_formula(a, b, c) == moves_by_plans(a, b, c) == sol.numMovesStones(a, b, c)
print("   và 20000 bộ ngẫu nhiên (thứ tự bất kỳ): các dãy nước đi dựng được đều hợp lệ và có độ dài min/max")

# 12.5 tính chất: đối xứng theo hoán vị và phản chiếu; tịnh tiến; min <= max; max = z - x - 2; min in {0,1,2}
rng = random.Random(5033)
for _ in range(3000):
    a, b, c = rng.sample(range(1, 80), 3)
    r = sol.numMovesStones(a, b, c)
    assert r[0] <= r[1] and r[0] in (0, 1, 2) and r[1] == max(a, b, c) - min(a, b, c) - 2
    assert sol.numMovesStones(c, a, b) == sol.numMovesStones(b, c, a) == r                          # hoán vị
    assert sol.numMovesStones(101 - a, 101 - b, 101 - c) == r                                        # phản chiếu
    assert sol.numMovesStones(a + 15, b + 15, c + 15) == r                                           # tịnh tiến
    assert (r[1] == 0) == (r[0] == 0)                                                               # min = 0 <=> max = 0
print("12.5 tính chất: OK")

12.4 toàn miền [1, 100]: 161700 bộ tổ hợp: công thức = mô phỏng = dãy nước đi


   và 20000 bộ ngẫu nhiên (thứ tự bất kỳ): các dãy nước đi dựng được đều hợp lệ và có độ dài min/max
12.5 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp `a, b, c`

Đề cho ba số theo thứ tự bất kỳ (ví dụ 2 và 3). Coi `a < b < c` cho kết quả sai khi đầu vào không tăng dần.

In [21]:
a, b, c = counters["quên sắp xếp"]
print(f"({a},{b},{c}): sai ->", wrong_no_sort(a, b, c), "| đúng ->", Solution().numMovesStones(a, b, c))
assert wrong_no_sort(a, b, c) != Solution().numMovesStones(a, b, c)
print("ví dụ 3 (3, 5, 1): sai ->", wrong_no_sort(3, 5, 1), "| đúng ->", Solution().numMovesStones(3, 5, 1))

(1,3,2): sai -> [1, -1] | đúng -> [0, 0]
ví dụ 3 (3, 5, 1): sai -> [1, -4] | đúng -> [1, 2]


Vì sao sai: `z - x - 2` với `z = 1`, `x = 3` cho số âm. **Đúng:** `x, y, z = sorted((a, b, c))`.

### Lỗi 2 — Nước đi nhiều nhất lệch một đơn vị

`max = z - x - 2`: khi `z - x = 2` (liên tiếp) phải ra `0`. Dùng `z - x - 1` cho `1`.

In [22]:
a, b, c = counters["max lệch một đơn vị"]
print(f"({a},{b},{c}): sai ->", wrong_max_off_by_one(a, b, c), "| đúng ->", Solution().numMovesStones(a, b, c))
assert wrong_max_off_by_one(a, b, c) != Solution().numMovesStones(a, b, c)

(1,2,3): sai -> [0, 1] | đúng -> [0, 0]


Vì sao sai: khi đã liên tiếp không có nước đi nào. **Đúng:** `z - x - 2` (số vị trí trống giữa hai đầu mút).

### Lỗi 3 — Chỉ xét khoảng trống bên trái

Nước đi ít nhất `1` cần **một trong hai** khoảng trống nhỏ, bên trái **hoặc** bên phải.

In [23]:
a, b, c = counters["chỉ xét khoảng trống bên trái"]
print(f"({a},{b},{c}): sai ->", wrong_left_gap_only(a, b, c), "| đúng ->", Solution().numMovesStones(a, b, c))
assert wrong_left_gap_only(a, b, c) != Solution().numMovesStones(a, b, c)

(1,4,5): sai -> [2, 2] | đúng -> [1, 2]


Vì sao sai: `(1, 5, 6)` có `z - y = 1` nhưng `y - x = 4`; bỏ qua bên phải cho `2` thay vì `1`. **Đúng:** `y - x <= 2 or z - y <= 2`.

### Lỗi 4 — Dùng `< 2` thay vì `<= 2`

Khoảng cách `2` nghĩa là có **đúng một** ô trống: đặt viên còn lại vào đó, `1` nước.

In [24]:
a, b, c = counters["dùng < 2 thay vì <= 2"]
print(f"({a},{b},{c}): sai ->", wrong_strict_less(a, b, c), "| đúng ->", Solution().numMovesStones(a, b, c))
assert wrong_strict_less(a, b, c) != Solution().numMovesStones(a, b, c)

(1,3,5): sai -> [2, 2] | đúng -> [1, 2]


Vì sao sai: `y - x = 2` (một ô trống giữa `x` và `y`) vẫn xong trong một nước. **Đúng:** `<= 2`.

### Lỗi 5 — Sai thứ tự kiểm tra các nhánh, hoặc trả về `[max, min]`

Nếu xét nhánh "`1`" trước "đã liên tiếp", `(1, 2, 3)` có `y - x = 1 <= 2` nên cho `1` thay vì `0`. Ngoài ra đề yêu cầu `[ít nhất, nhiều nhất]`.

In [25]:
a, b, c = counters["xét nhánh 1 trước nhánh đã liên tiếp"]
print(f"({a},{b},{c}): sai ->", wrong_consecutive_last(a, b, c), "| đúng ->", Solution().numMovesStones(a, b, c))
assert wrong_consecutive_last(a, b, c) != Solution().numMovesStones(a, b, c)
a, b, c = counters["trả về [max, min]"]
print(f"({a},{b},{c}): sai ->", wrong_swapped(a, b, c), "| đúng ->", Solution().numMovesStones(a, b, c))
assert wrong_swapped(a, b, c) != Solution().numMovesStones(a, b, c)

(1,2,3): sai -> [1, 0] | đúng -> [0, 0]
(1,2,5): sai -> [2, 1] | đúng -> [1, 2]


Vì sao sai: nhánh đã liên tiếp phải đứng **trước**; và `answer[0]` là số ít nhất. **Đúng:** kiểm tra `z - x == 2` trước, trả `[least, most]`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "trò chơi di chuyển với luật cố định trên vài đối tượng, hỏi ít nhất và nhiều nhất" ⇒ thử **brute force** (cây trò chơi có nhớ) với dữ liệu nhỏ, quan sát bảng đáp án rồi **đoán và chứng minh công thức**.
- Nhìn bài toán qua **khoảng cách** hay **số ô trống** giữa các đối tượng, không phải vị trí tuyệt đối.
- Tìm **đại lượng giảm đơn điệu** (ở đây `z - x`) để chặn số nước đi tối đa.
- Cận dưới nhỏ (`0`, `1`, `2`) thường tới từ một vài **trường hợp** nhỏ: liệt kê và chứng minh từng trường hợp.
- Luôn **sắp xếp** đầu vào khi bài chỉ phụ thuộc thứ tự tương đối.
- Cùng họ: Moving Stones Until Consecutive II (1040), Stone Game (877), Nim Game (292).
- Test nên có: đã liên tiếp, một khoảng trống bằng `0`/`1`/`2`/`3`, hai khoảng trống lớn, thứ tự đầu vào bất kỳ, biên `1` và `100`.

## 15. Biến thể

1. **Phiên bản II (1040):** `n` viên đá, mỗi nước nhấc một viên **đầu mút** và đặt vào vị trí **ngoài** (không còn là đầu mút); cần cửa sổ trượt.
2. **Cho phép đặt ngoài đoạn** hoặc nhấc viên bất kỳ.
3. **In một dãy nước đi** ít nhất/nhiều nhất: Cách 4 (bài tập 5).
4. Bài liên quan: 1040, 877.

Biến thể: đếm số **trạng thái cuối khác nhau** có thể đạt (vị trí tập ba viên liên tiếp) bằng oracle: kiểm bằng cây trò chơi.

In [26]:
@lru_cache(None)
def final_states(x: int, y: int, z: int):
    if z - x == 2:
        return frozenset({(x, y, z)})
    out = set()
    for src in (x, z):
        for k in range(x + 1, z):
            if k != y:
                rest = tuple(sorted(({x, y, z} - {src}) | {k}))
                out |= final_states(*rest)
    return frozenset(out)

finals = final_states(1, 4, 9)
print("trạng thái cuối đạt được từ (1, 4, 9):", sorted(finals))
# các trạng thái cuối nằm trong [1, 9], độ rộng 2: dạng (t, t+1, t+2) với 1 <= t <= 7
assert all(s[1] - s[0] == 1 and s[2] - s[1] == 1 and 1 <= s[0] and s[2] <= 9 for s in finals)
print("số trạng thái cuối:", len(finals))

trạng thái cuối đạt được từ (1, 4, 9): [(1, 2, 3), (2, 3, 4), (3, 4, 5), (4, 5, 6), (5, 6, 7), (6, 7, 8), (7, 8, 9)]
số trạng thái cuối: 7


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cây trò chơi có nhớ | Xét mọi nước đi | `O(N^4)` | `O(N^3)` | Oracle, miền nhỏ |
| 2. Mô phỏng nước chậm nhất | Đếm `max` bằng cách chơi | `O(z - x)` | `O(1)` | Hiểu cơ chế |
| 3. Công thức | `z - x - 2` và 3 trường hợp `min` | `O(1)` | `O(1)` | **Cách nộp chính** |
| 4. Dựng dãy nước đi | Chứng minh đạt được | `O(z - x)` | `O(z - x)` | Kiểm tra, xuất dãy |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(sorted((3, 5, 1)), 5 - 1 - 2)
print(Solution().numMovesStones(2, 3, 4), Solution().numMovesStones(1, 3, 4))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # danh sách đã sắp xếp và một số
my_guess_2 = None      # hai danh sách [min, max]

### Bài 2 — Tính tay (dễ-vừa)

`(a, b, c) = (1, 4, 9)`. Ghi `[ít nhất, nhiều nhất]`.

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành công thức.

In [29]:
def exercise_3(a: int, b: int, c: int) -> List[int]:
    # TODO 1: sắp xếp thành x < y < z
    # TODO 2: tính least theo thứ tự nhánh: đã liên tiếp -> 0; một khoảng cách <= 2 -> 1; còn lại -> 2
    # TODO 3: trả về [least, z - x - 2]
    pass


# for (a, b, c), expected in examples:
#     assert exercise_3(a, b, c) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(a, b, c)` theo **hai khoảng trống** `g1 = y - x - 1`, `g2 = z - y - 1`: `max = g1 + g2`; `min = 0` nếu `g1 = g2 = 0`, `1` nếu `min(g1, g2) <= 1`, còn lại `2`.

In [30]:
def exercise_4(a: int, b: int, c: int) -> List[int]:
    # TODO
    pass


# for (a, b, c), expected in examples:
#     assert exercise_4(a, b, c) == expected

### Bài 5 — Mini challenge: dãy nước đi ít nhất (khó)

Viết `exercise_5(a, b, c)` trả về một **dãy nước đi** (danh sách các cặp `(vị_trí_nhấc, vị_trí_đặt)`) có độ dài **đúng bằng số nước đi ít nhất**, hợp lệ theo luật của đề và kết thúc với ba viên liên tiếp. Có nhiều dãy hợp lệ; ta kiểm tra bằng bộ mô phỏng `run_plan` (không so sánh với một dãy cố định).

```python
exercise_5(1, 2, 5)  -> độ dài 1, ví dụ [(5, 3)]
exercise_5(1, 4, 9)  -> độ dài 2
exercise_5(2, 3, 4)  -> []
```
Gợi ý: xét các trường hợp của mục 6.

In [31]:
def exercise_5(a: int, b: int, c: int) -> List[Tuple[int, int]]:
    # TODO
    pass


# for t in [(1, 2, 5), (1, 4, 9), (2, 3, 4), (3, 5, 1)]:
#     plan = exercise_5(*t)
#     assert run_plan(*t, plan) == moves_formula(*t)[0]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `sorted` trả danh sách tăng dần; `z - x - 2` với `z = 5`, `x = 1`.
- Mức 2: `(2, 3, 4)` đã liên tiếp; `(1, 3, 4)`: `x = 1`, `y = 3`, `z = 4`, `z - x = 3`.
- Mức 3: dòng 1: `[1, 3, 5] 2`; dòng 2: `[0, 0] [1, 1]`.

**Bài 2**
- Mức 1: `x = 1`, `y = 4`, `z = 9`; `y - x = 3`, `z - y = 5`.
- Mức 2: không có khoảng cách `<= 2` và chưa liên tiếp: `min = 2`; `max = 9 - 1 - 2 = 6`.
- Mức 3: `[2, 6]`.

**Bài 3**
- Mức 1: `x, y, z = sorted((a, b, c))`.
- Mức 2: `if z - x == 2: least = 0; elif y - x <= 2 or z - y <= 2: least = 1; else: least = 2`.
- Mức 3: `return [least, z - x - 2]`.

**Bài 4**
- Mức 1: `g1 = y - x - 1`, `g2 = z - y - 1`.
- Mức 2: `max = g1 + g2`; `min = 0` nếu cả hai bằng `0`.
- Mức 3: `min = 1` nếu `min(g1, g2) <= 1` (ngược lại `2`).

**Bài 5**
- Mức 1: `y - x == 2` (một ô trống bên trái): đặt `z` vào `x + 1`.
- Mức 2: `y - x == 1`: đặt `z` vào `y + 1`; các trường hợp bên phải đối xứng.
- Mức 3: cả hai khoảng trống `>= 2`: `[(x, y - 1), (z, y + 1)]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(sorted((3, 5, 1)), 5 - 1 - 2)
print(Solution().numMovesStones(2, 3, 4), Solution().numMovesStones(1, 3, 4))

assert (sorted((3, 5, 1)), 5 - 1 - 2) == ([1, 3, 5], 2)
assert (Solution().numMovesStones(2, 3, 4), Solution().numMovesStones(1, 3, 4)) == ([0, 0], [1, 1])

[1, 3, 5] 2
[0, 0] [1, 1]


### Đáp án Bài 2

`(1, 4, 9)`: hai khoảng cách `3` và `5` đều `> 2`, chưa liên tiếp: `min = 2`; `max = 9 - 1 - 2 = 6`.

In [33]:
hand_answer_solution = [2, 6]
assert Solution().numMovesStones(1, 4, 9) == hand_answer_solution == moves_brute(1, 4, 9)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_solution(a: int, b: int, c: int) -> List[int]:
    x, y, z = sorted((a, b, c))
    if z - x == 2:
        least = 0
    elif y - x <= 2 or z - y <= 2:
        least = 1
    else:
        least = 2
    return [least, z - x - 2]

for (a, b, c), expected in examples:
    assert exercise_3_solution(a, b, c) == expected
for combo in combinations(range(1, 16), 3):
    assert exercise_3_solution(*combo) == moves_brute(*combo)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_solution(a: int, b: int, c: int) -> List[int]:
    x, y, z = sorted((a, b, c))
    g1, g2 = y - x - 1, z - y - 1
    if g1 == 0 and g2 == 0:
        least = 0
    elif min(g1, g2) <= 1:
        least = 1
    else:
        least = 2
    return [least, g1 + g2]

for (a, b, c), expected in examples:
    assert exercise_4_solution(a, b, c) == expected
for combo in combinations(range(1, 16), 3):
    assert exercise_4_solution(*combo) == moves_brute(*combo)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_solution(a: int, b: int, c: int) -> List[Tuple[int, int]]:
    x, y, z = sorted((a, b, c))
    if z - x == 2:
        return []
    if y - x == 2:
        return [(z, x + 1)]
    if y - x == 1:
        return [(z, y + 1)]
    if z - y == 2:
        return [(x, y + 1)]
    if z - y == 1:
        return [(x, y - 1)]
    return [(x, y - 1), (z, y + 1)]

for t in [(1, 2, 5), (1, 4, 9), (2, 3, 4), (3, 5, 1)]:
    plan = exercise_5_solution(*t)
    assert run_plan(*t, plan) == moves_formula(*t)[0]
for combo in combinations(range(1, 21), 3):
    assert run_plan(*combo, exercise_5_solution(*combo)) == moves_brute(*combo)[0]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (công thức `O(1)`)

```python
class Solution:
    def numMovesStones(self, a: int, b: int, c: int) -> List[int]:
        x, y, z = sorted((a, b, c))                 # đưa về x < y < z
        if z - x == 2:                              # đã liên tiếp
            least = 0
        elif y - x <= 2 or z - y <= 2:              # một khoảng trống bằng 0 hoặc 1
            least = 1
        else:
            least = 2
        return [least, z - x - 2]
```

Công thức nhớ nhanh: `max = z - x - 2`; `min = 0` nếu đã liên tiếp, `1` nếu một trong hai khoảng cách `y - x`, `z - y` không vượt quá `2`, còn lại `2`.

### Checklist trước khi submit

- [ ] Đã `sorted((a, b, c))` trước khi tính?
- [ ] `max = z - x - 2` (không `- 1`)?
- [ ] Kiểm tra "đã liên tiếp" **trước** các nhánh khác?
- [ ] Xét **cả hai** khoảng cách `y - x` và `z - y`, và so với `<= 2` (không `< 2`)?
- [ ] Trả `[ít nhất, nhiều nhất]` đúng thứ tự?
- [ ] Đã thử: `(1,2,3)`, `(1,2,4)`, `(1,3,5)`, `(1,4,9)`, thứ tự đầu vào bất kỳ, `(1,50,100)`?